# Train LightGCN with PyTorch

This notebook turns the NumPy reference into trainable embedding parameters. It uses synthetic data only; decreasing loss here is a learning sanity check, not recommendation performance.

In [ ]:
from dataclasses import asdict
import json
from pathlib import Path
import sys

import pandas as pd
import torch

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from group_movie_recommender.algorithms.graph_data import BPRBatchSampler
from group_movie_recommender.algorithms.lightgcn import LightGCN
from group_movie_recommender.pipelines.lightgcn_training import (
    LightGCNTrainingConfig,
    batch_to_tensors,
    build_toy_training_graph,
    train_small_graph,
)

## 1. Initialize trainable node embeddings

The graph and normalization weights are fixed. Only the layer-zero embedding table is an optimized parameter.

In [ ]:
config_values = json.loads(
    (project_root / "configs/lightgcn_toy.json").read_text(encoding="utf-8")
)
config = LightGCNTrainingConfig(**config_values)
graph = build_toy_training_graph()
model = LightGCN(
    graph,
    embedding_dim=config.embedding_dim,
    num_layers=config.num_layers,
    random_seed=config.random_seed,
)

print(asdict(config))
print("Embedding table shape:", tuple(model.embedding.weight.shape))
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

## 2. Follow one gradient

Automatic differentiation connects BPR loss back through sparse propagation to the initial embedding table.

In [ ]:
sampler = BPRBatchSampler(graph, random_seed=config.random_seed + 1)
batch = sampler.sample(batch_size=config.batch_size)
batch_tensors = batch_to_tensors(batch)
losses = model.bpr_objective(
    *batch_tensors,
    l2_weight=config.l2_weight,
)
losses["loss"].backward()

print({name: float(value.detach()) for name, value in losses.items()})
print("Gradient exists:", model.embedding.weight.grad is not None)
print("Gradient magnitude:", float(model.embedding.weight.grad.abs().sum()))

## 3. Optimize fresh mini-batches

The trainer monitors one fixed training batch only to confirm that learning occurs. Hyperparameters must later be selected with validation rankings, not this loss.

In [ ]:
trained_model, history = train_small_graph(graph, config)
display(history.iloc[::max(1, config.steps // 5)])

initial_loss = history.iloc[0]["fixed_training_bpr_loss"]
final_loss = history.iloc[-1]["fixed_training_bpr_loss"]
print(f"Fixed training BPR loss: {initial_loss:.6f} -> {final_loss:.6f}")

## 4. Produce individual full-catalog scores

For one user, a matrix-vector product scores every movie. Previously seen movies are marked here and must be removed before ranking.

In [ ]:
with torch.no_grad():
    user_embeddings, movie_embeddings = trained_model.propagate()
    user_index = 0
    all_scores = movie_embeddings @ user_embeddings[user_index]

seen_local_movies = set(
    graph.positive_movie_indices[graph.positive_user_indices == user_index].tolist()
)
catalog_scores = pd.DataFrame(
    {
        "movieIndex": range(graph.num_movies),
        "movieId": graph.movie_ids,
        "score": all_scores.numpy(),
    }
)
catalog_scores["seenInTrain"] = catalog_scores["movieIndex"].isin(seen_local_movies)
catalog_scores.sort_values("score", ascending=False)

## Next step

Train on a controlled MovieLens subset, score warm unseen candidates for each pair member, and connect those individual scores to average and conflict-aware group ranking.